In [9]:
#준비 셀 1. CIFAR-10과 실험실
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Subset, TensorDataset
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True # cuDNN도 결정적 계산만
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225) # ImageNet 사진의 평균, 표준편차
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
train_all = datasets.CIFAR10(root="data", train=True, download=True, transform=tf)
g = torch.Generator().manual_seed(0) # 학습과 검증 분리, seed 0
train_set, val_set = random_split(train_all, [45000, 5000], generator=g)
lab = Subset(train_all, train_set.indices[:5000]) # 실험실 5,000장
xs = torch.stack([img for img, _ in lab])
ys = torch.tensor([y for _, y in lab])
xv = torch.stack([img for img, _ in val_set]) # 검증 5,000장
yv = torch.tensor([y for _, y in val_set])
lab_loader = DataLoader(TensorDataset(xs, ys), batch_size=64, shuffle=True)
val_fast = DataLoader(TensorDataset(xv, yv), batch_size=256, shuffle=False)
print(xs.shape, len(lab_loader), len(val_fast)) # → torch.Size([5000, 3, 32, 32]) 79 20

torch.Size([5000, 3, 32, 32]) 79 20


In [10]:
#train_one_epoch의 앞부분, 더한 세 줄
def train_one_epoch(model, loader, opt):
 model.train()
 for m in model.modules(): # 모든 층을 하나씩 꺼내
    if isinstance(m, nn.BatchNorm2d) and not m.weight.requires_grad:
        m.eval() # 고정한 BN은 평균, 분산 기록도 그대로
 ... # 아래는 지난시간과 같다

In [ ]:
import time
import torch
import torch.nn as nn

def make_cnn(widths=(16, 32)): 
    """CNN 모델 생성 함수"""
    layers, cin, size = [], 3, 32
    for w in widths:
        layers += [
            nn.Conv2d(cin, w, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        ]
        cin, size = w, size // 2  # 루프 안에서 입력 채널과 크기 업데이트
    
    # 루프가 끝난 후 Flatten과 Linear 레이어 추가
    layers += [nn.Flatten(), nn.Linear(cin * size * size, 10)]
    return nn.Sequential(*layers).to(device)

def count_train(model): 
    """학습 가능한 파라미터 수 계산"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def run(model, opt, loader=lab_loader, epochs=5): 
    """실험 실행 함수"""
    print("train", count_train(model), end=" ")
    t0 = time.time()
    
    # 주의: 여기서 fit 함수가 반드시 정의되어 있어야 합니다!
    result = fit(model, loader, epochs, opt) 
    
    
    print(" time", round(time.time() - t0, 1)) # 걸린 시간(초)
    return result


In [12]:
#예제 9-1. 지난시간 make_cnn으로 블록 3개, 처음부터 5 epoch
torch.manual_seed(0)
model = make_cnn((32, 64, 128)) # 블록 3개 CNN, 처음부터 학습
opt = torch.optim.AdamW(model.parameters(), lr=0.001)
result = run(model, opt)
# → (예) train 113738 best 4 54.84 last 54.24
# → (예) time 2.5

train 113738 

NameError: name 'fit' is not defined